# Prova dataset

In [2]:
import pandas as pd

data = pd.read_csv('amazon_delivery_OLD.csv')
print(data.head())

        Order_ID  Agent_Age  Agent_Rating  Store_Latitude  Store_Longitude  \
0  ialx566343618         37           4.9       22.745049        75.892471   
1  akqg208421122         34           4.5       12.913041        77.683237   
2  njpu434582536         23           4.4       12.914264        77.678400   
3  rjto796129700         38           4.7       11.003669        76.976494   
4  zguw716275638         32           4.6       12.972793        80.249982   

   Drop_Latitude  Drop_Longitude  Order_Date Order_Time Pickup_Time  \
0      22.765049       75.912471  2022-03-19   11:30:00    11:45:00   
1      13.043041       77.813237  2022-03-25   19:45:00    19:50:00   
2      12.924264       77.688400  2022-03-19   08:30:00    08:45:00   
3      11.053669       77.026494  2022-04-05   18:00:00    18:10:00   
4      13.012793       80.289982  2022-03-26   13:30:00    13:45:00   

      Weather  Traffic      Vehicle            Area  Delivery_Time  \
0       Sunny    High   motorcycle

# Ricerca in Spazi di Stati (Search)
## Pre Processing


- Verificare qual’è la latitudine e longitudine da cui parte il maggior numero di ordini
- Una volta individuato, verificare che ci siano per questo magazzino, più ordini per la stessa destinazione. Saranno i record da considerare.

In [3]:
import pandas as pd

# Carica il dataset
df = pd.read_csv('amazon_delivery_OLD.csv')

print("=== DATASET ORIGINALE ===")
print(f"Dimensioni: {df.shape}")
print(df.head())

# Rimuovo i record con coordinate nulle o zero (dati mancanti/errati)
df_clean = df[(df['Store_Latitude'] != 0.0) & 
              (df['Store_Longitude'] != 0.0) & 
              (df['Drop_Latitude'] != 0.0) & 
              (df['Drop_Longitude'] != 0.0)].copy()

print(f"\n=== PULIZIA DATI ===")
print(f"Record originali: {len(df)}")
print(f"Record con coordinate valide: {len(df_clean)}")
print(f"Record rimossi (coordinate nulle): {len(df) - len(df_clean)}")

# Creo identificatori univoci per magazzini di partenza e destinazione
df_clean['Store_ID'] = df_clean['Store_Latitude'].round(4).astype(str) + '_' + df_clean['Store_Longitude'].round(4).astype(str)
df_clean['Destination_ID'] = df_clean['Drop_Latitude'].round(4).astype(str) + '_' + df_clean['Drop_Longitude'].round(4).astype(str)

# Conto il numero di ordini per ogni combinazione Store -> Destination
conteggio_rotte = df_clean.groupby(['Store_ID', 'Destination_ID']).size().reset_index(name='Conteggio_Rotta')

# Trovo la rotta con il maggior numero di ordini
rotta_principale = conteggio_rotte.loc[conteggio_rotte['Conteggio_Rotta'].idxmax()]

print("\n=== ROTTA CON MAGGIOR NUMERO DI ORDINI ===")
print(f"Magazzino partenza (Store_ID): {rotta_principale['Store_ID']}")
print(f"Destinazione (Destination_ID): {rotta_principale['Destination_ID']}")
print(f"Numero ordini: {rotta_principale['Conteggio_Rotta']}")

# Filtro il dataset per ottenere tutti i record di questa rotta
store_principale = rotta_principale['Store_ID']
destinazione_principale = rotta_principale['Destination_ID']

dataset_filtrato = df_clean[(df_clean['Store_ID'] == store_principale) & 
                             (df_clean['Destination_ID'] == destinazione_principale)].copy()

# Rimuovo le colonne ausiliarie create per il raggruppamento
dataset_filtrato = dataset_filtrato.drop(columns=['Store_ID', 'Destination_ID'])

print("\n=== DATASET FILTRATO ===")
print(f"Numero di record: {len(dataset_filtrato)}")
print(f"\nColonne mantenute: {dataset_filtrato.columns.tolist()}")
print(f"\nCoordinate Magazzino: ({dataset_filtrato['Store_Latitude'].iloc[0]}, {dataset_filtrato['Store_Longitude'].iloc[0]})")
print(f"Coordinate Destinazione: ({dataset_filtrato['Drop_Latitude'].iloc[0]}, {dataset_filtrato['Drop_Longitude'].iloc[0]})")
print("\nAnteprima dati:")
print(dataset_filtrato)

# Salvo il dataset filtrato in CSV
dataset_filtrato.to_csv('rotta_principale_ordini.csv', index=False)

print("\n=== FILE SALVATO ===")
print("File CSV salvato: 'rotta_principale_ordini.csv'")
print(f"Record totali: {len(dataset_filtrato)}")

=== DATASET ORIGINALE ===
Dimensioni: (43648, 20)
        Order_ID  Agent_Age  Agent_Rating  Store_Latitude  Store_Longitude  \
0  ialx566343618         37           4.9       22.745049        75.892471   
1  akqg208421122         34           4.5       12.913041        77.683237   
2  njpu434582536         23           4.4       12.914264        77.678400   
3  rjto796129700         38           4.7       11.003669        76.976494   
4  zguw716275638         32           4.6       12.972793        80.249982   

   Drop_Latitude  Drop_Longitude  Order_Date Order_Time Pickup_Time  \
0      22.765049       75.912471  2022-03-19   11:30:00    11:45:00   
1      13.043041       77.813237  2022-03-25   19:45:00    19:50:00   
2      12.924264       77.688400  2022-03-19   08:30:00    08:45:00   
3      11.053669       77.026494  2022-04-05   18:00:00    18:10:00   
4      13.012793       80.289982  2022-03-26   13:30:00    13:45:00   

      Weather  Traffic      Vehicle            Area  D